In [1]:
!nvidia-smi

Thu Jul  9 15:32:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:19:00.0 Off |                   On |
| N/A   31C    P0             70W /  300W |    1240MiB /  81920MiB |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

|  0    2   0   0  |             107MiB / 40192MiB    | 42      0 |  3   0    2    0    0 |
|                  |               0MiB / 24543MiB    |           |                       |
+------------------+----------------------------------+-----------+-----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|    1    2    0          2165345      C   /usr/local/bin/python3.12               574MiB |
|    1    1    0          2889512      C   /opt/venv/bin/python3                  3570MiB |
|    2    1    0          2978869      C   /usr/local/bin/python               

In [2]:
import xarray as xr

# path = "/home/orabe/fNIRS_sparseToDense/datasets_14062026/processed/vfc_hd/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs_rest_4_test.nc"
# path = "datasets_14062026/processed/Anderson_sparse/sub-1/sub-1_ses-1_task-WordStroop_run-1_nirs_rest_4_test.nc"
path = "datasets/processed/imageRecon_params/vfc_hd/full/am_1__as_1/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs_rest_0.nc"

data = xr.open_dataarray(path)
data.sizes


Frozen({'parcel': 143, 'chromo': 2, 'time': 174})

In [3]:
import os
import numpy as np
import xarray as xr
import pickle
import glob
import warnings
from pathlib import Path, PureWindowsPath
import matplotlib.pyplot as plt
import re

import cedalion
import cedalion.sigproc.motion as motion_correct
import cedalion.sigproc.quality as quality
import cedalion.sigproc.physio as physio
import cedalion.dot as dot
# NEW: helper-only alpha_meas estimation API for recon augmentation
import cedalion.dot.image_recon as dot_image_recon
import cedalion.nirs as nirs
import cedalion.vis.anatomy
from cedalion.io.forward_model import load_Adot

from cedalion import units
import pandas as pd

warnings.filterwarnings("ignore")

In [4]:
def get_bad_ch_mask(int_data, ch_preproc) -> list:
    # Saturated and Dark Channels

    dark_sat_thresh = [1e-3, 0.84]
    amp_threshs_sat = [0., dark_sat_thresh[1]]
    amp_threshs_low = [dark_sat_thresh[0], 1]
    _, amp_mask_sat = quality.mean_amp(int_data, amp_threshs_sat)
    _, amp_mask_low = quality.mean_amp(int_data, amp_threshs_low)
    _, snr_mask = quality.snr(int_data, 10)
    amp_mask=amp_mask_sat & amp_mask_low

    _, list_bad_ch = quality.prune_ch(int_data, [amp_mask, snr_mask], "all")
   
    return list_bad_ch

# Theekshana
# def get_bad_ch_mask(int_data: xr.DataArray, ch_preproc: dict) -> list:
#     # SCI and PSP Mask    
#     sci, sci_mask = quality.sci(int_data, ch_preproc['window_len'], ch_preproc['sci_thresh'])
#     psp, psp_mask = quality.psp(int_data, ch_preproc['window_len'], ch_preproc['psp_thresh'])

#     sci_psp_mask=sci_mask & psp_mask
#     perc_time_clean = sci_psp_mask.sum(dim="time") / len(sci.time)

#     scipsp_bad_ch=[]
#     for ch in perc_time_clean.channel.values:
#         if perc_time_clean.sel(channel=ch).values < ch_preproc['perc_time_clean']: # dont make the mistake of using the inv condition >:| 
#             scipsp_bad_ch.append(ch)

#     sum_bad_ch = scipsp_bad_ch
#     list_bad_ch = sorted(list(set(sum_bad_ch)))  # remove duplicates

#     print("Flagged Channels : ",len(list_bad_ch), '/', len(int_data.channel))
#     print("Percentage: ", int(len(list_bad_ch) / len(int_data.channel) * 100), '%')
    
#     return list_bad_ch

In [5]:
def standardize_trial_types(DATASET_NAME: str, file: str, stim: pd.DataFrame, rec):
    
    if DATASET_NAME == "FreshMotor":
        # map trial types to left or right depending on the name of the file
        m = re.search(r'(?i)(left|right)', file)

        # rename from MOTOR to left/right
        rec.stim.trial_type = m.group(1).lower()
        # rec.stim = stim
    
    elif DATASET_NAME == "BallSqueezingHD":
        mapping = {
            "Right": "right", # BallSqueezingHD
            "Left": "left",   # BallSqueezingHD
        }
        rec.stim["trial_type"] = rec.stim["trial_type"].replace(mapping)
        
    elif DATASET_NAME == "BS_Laura":
        stim = stim.copy()
        # stim["duration"] = 10.0  # NEW FIX: BS_Laura events are treated as 10s, not the raw 5s annotation
        rec.stim = stim
        
    elif DATASET_NAME == "Electrical_Thermal":
        mapping = {
            "1": "WordCongruent" ,
            "2": "WordIncongruent",
        }
        rec.stim["trial_type"] = rec.stim["trial_type"].replace(mapping)
        
    elif DATASET_NAME in ["vfc_hd", "Anderson_sparse"]:
        mapping = {
            "1": "WordCongruent" ,
            "2": "WordIncongruent",
        }
        
        rec.stim["trial_type"] = rec.stim["trial_type"].replace(mapping)
    
    rec.stim.sort_values(by="onset", ignore_index=True, inplace=True)

    # attach/update stim info to rec
    # rec.stim = stim

    return stim, rec

# def standardize_trial_types(DATASET_NAME: str, file: str, rec):
    
#     if DATASET_NAME == "FreshMotor":
#         # map trial types to left or right depending on the name of the file
#         m = re.search(r'(?i)(left|right)', file)

#         # rename from MOTOR to left/right
#         rec.stim.trial_type = m.group(1).lower()
    
#     else:
#         mapping = {
#             "Right": "right", # BallSqueezingHD
#             "Left": "left",   # BallSqueezingHD
#             "ElectricalVAS7": "right", # TODO: Electrical_Thermal
#             "ElectricalVAS3": "left",  # TODO: Electrical_Thermal
#         }
#         rec.stim["trial_type"] = rec.stim["trial_type"].replace(mapping)

#     return rec


In [ ]:
# "full", "subset_2" for spatial_sampling, "motor" for motor_sampling

# subset_type = ""


In [7]:
# augmentation_strategy = "imageRecon_params"
augmentation_strategy = "unmodified"

mixed_sparse = False
if mixed_sparse:
    n_chs = 50
    subset_type = f"motor_{n_chs}chs" # for sparsified version of Laura
    sparsified_data_path = f'datasets/pre_processed/BS_Laura/channel_subset_BS_Laura_k=2_c3c4k=51_dist4.5_{n_chs}chs.npy'
else:
    subset_type = "full"

base_path = "/home/orabe/fNIRS_sparseToDense/"

# Available datasets:
# DATASET_NAME = "BallSqueezingHD_modified"
# DATASET_NAME = "FreshMotor"
DATASET_NAME = "BS_Laura"
# DATASET_NAME = "ElectricalThermal"
# DATASET_NAME = "vfc_hd"
# DATASET_NAME = "Anderson_sparse"

raw_path = Path(f'datasets/raw/{DATASET_NAME}')

pre_processed_path = Path(f'datasets/pre_processed/{augmentation_strategy}/{DATASET_NAME}/{subset_type}')

pre_processed_path.mkdir(parents=True, exist_ok=True)
pre_processed_path

# # For Laura channel space subset 2
# pre_processed_path = Path(f'datasets/pre_processed/imageRecon_params/BS_Laura/subset_2/motor_100chs')
# pre_processed_path.mkdir(parents=True, exist_ok=True)
# pre_processed_path


PosixPath('datasets/pre_processed/unmodified/BS_Laura/full')

In [25]:
# --- NEW: recon parameter augmentation config ---
k_alpha_meas = 0.01
alpha_meas_multipliers = [0.1, 1.0, 10.0]
alpha_spatial_multipliers = [0.1, 1.0, 10.0]
baseline_alpha_spatial = 1e-2


def get_recon_settings(c_meas):
    try:
        alpha_meas_0 = dot_image_recon.estimate_alpha_meas(c_meas, K=k_alpha_meas)
    except Exception:
        # fallback for older Cedalion versions: alpha_meas = K / median(C_meas)
        alpha_meas_0 = k_alpha_meas / np.median(c_meas.values)

    alpha_spatial_0 = float(baseline_alpha_spatial)
    settings = []
    for m_meas in alpha_meas_multipliers:
        for m_spatial in alpha_spatial_multipliers:
            settings.append(
                (
                    f"am_{m_meas:g}__as_{m_spatial:g}",
                    float(alpha_meas_0 * m_meas),
                    float(alpha_spatial_0 * m_spatial),
                    alpha_meas_0,
                    alpha_spatial_0,
                    float(m_meas),
                    float(m_spatial),
                )
            )

    return settings


def get_param_config_folder(alpha_meas_multiplier, alpha_spatial_multiplier):
    return f"am_{float(alpha_meas_multiplier):g}__as_{float(alpha_spatial_multiplier):g}"

pre_processed_path


PosixPath('datasets/pre_processed/imageRecon_params/BS_Laura/full')

In [26]:
if DATASET_NAME == "BallSqueezingHD_modified":
    raw_dir = f"{raw_path}/sub-*/nirs/sub-*.snirf"

elif DATASET_NAME == "BS_Laura":
    raw_dir = f"{raw_path}/sub-*/nirs/sub-*.snirf"
    
elif DATASET_NAME == "Electrical_Thermal":
    raw_dir = f"{raw_path}/sub-*/ses-*/nirs/sub-*_ses-*_task-Electrical*_nirs.snirf"
    # TODO: exclude subjects without txt files for landmarks coords
    
elif DATASET_NAME == "FreshMotor":
    duration = "*" # * to include both 2s and 3s
    raw_dir = f"{raw_path}/sub-*/ses-*{duration}/nirs/sub-*_ses-*{duration}_task-FRESHMOTOR_nirs.snirf"
elif DATASET_NAME == "vfc_hd":
    # "datasets/raw/vfc_hd/sub-01/nirs/sub-01_ses-02_task-WordStroop_run-01_nirs.snirf"
    raw_dir = f"{raw_path}/sub-*/nirs/sub-*_ses-*_task-WordStroop_run-*_nirs.snirf"
elif DATASET_NAME == "Anderson_sparse":
    # raw_dir = f"{raw_path}/sub-*/ses-*/nirs/sub-*_ses-*_task-WordStroop_run-*_nirs.snirf"
    raw_dir = f"{raw_path}/sub-*/nirs/sub-*_ses-*_task-WordStroop_run-*_nirs.snirf"    

else:
    raise ValueError("Unknown dataset name")

files = glob.glob(raw_dir)

# TODO: to be confirmed
# remove non-BS files for Laura's dataset to avoid errors
if DATASET_NAME == "BS_Laura":
    files = [p for p in files if "BS" in os.path.basename(p)]
    # remove files that has this pattern in the name: _acq-4NN_nirs:
    files = [p for p in files if "_acq-4NN_nirs" not in os.path.basename(p)]
    
files = sorted(files)
print(f"{len(files)} files found.")

45 files found.


In [27]:
files[0]

'datasets/raw/BS_Laura/sub-568/nirs/sub-568_task-BS_run-01_nirs.snirf'

In [28]:
# spatial_sampling is the approach we use to subsample channels based on predifined spatial locations. See src/subset/optode_subsets.ipynb
spatial_sampling = False
if spatial_sampling:
    # This is a temoraly code to load subset channels. For FreshMotor we don't have subsets yet as we need all channels. Thus we do construct subset_channels manually and make it equal to all channels.
    if DATASET_NAME == "BallSqueezingHD_modified":
        with open(f"results/subset/{DATASET_NAME}/subsets_data.pkl", "rb") as f:
            subsets_data = pickle.load(f)
        subset_channels = subsets_data[subset_type]["all"]
        
    elif DATASET_NAME == "FreshMotor":
        # make subset_channels equal to all channels
        filename = files[0] # select one
        rec = cedalion.io.read_snirf(filename)[0]  # read snirf files
        all_channels = rec['amp']['channel'].values.tolist()
        subset_channels = all_channels

# motor_sampling is the approach we use to subsample channels based on motor area locations. See src/subset/sparsify_chs_from_sens.ipynb
motor_sampling = False
if motor_sampling:
    if DATASET_NAME == "BS_Laura":
        subset_channels = np.load(sparsified_data_path, allow_pickle=True)
        
no_subsampling = True
if no_subsampling:
     # make subset_channels equal to all channels
    filename = files[0] # select one
    print(filename)
    rec = cedalion.io.read_snirf(filename)[0]  # read snirf files
    all_channels = rec['amp']['channel'].values.tolist()
    subset_channels = all_channels
    
print(len(subset_channels))

datasets/raw/BS_Laura/sub-568/nirs/sub-568_task-BS_run-01_nirs.snirf
1220


In [29]:
filename = files[0] # select one
rec = cedalion.io.read_snirf(filename)[0]  # read snirf files

# subset the data
rec['amp'] = rec['amp'].sel(channel=subset_channels)

# subset measurement list
meas_list = rec._measurement_lists["amp"]
meas_list = meas_list[meas_list["channel"].isin(subset_channels)].reset_index(drop=True)

# both should then be equal
print(f'Number of channels in rec["amp"]: {len(set(rec["amp"].channel.values))}')
print(f'Number of channels in meas_list: {len(set(meas_list.channel.values))}')

head_icbm152 = dot.get_standard_headmodel('icbm152')  


if DATASET_NAME == "BS_Laura":
    # this is required for BU Data (Laura's)
    T = np.array([
        [-9.57882733e-01, -7.20806358e-03,  6.20193531e-03, 2.21208571e+02],
        [-2.02271710e-02,  6.03819925e-02,  9.94046165e-01, -2.03010603e+01],
        [-8.79481533e-03, -1.02761992e+00,  6.59199998e-02, 2.87749135e+02],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, 1.00000000e+00]])
    
    ninja_aligned = rec.geo3d.points.apply_transform(T)
    geo3d_snapped_ijk = head_icbm152.align_and_snap_to_scalp(ninja_aligned)
else:
    geo3d_snapped_ijk = head_icbm152.align_and_snap_to_scalp(rec.geo3d)
    

fwm = cedalion.dot.forward_model.ForwardModel(
    head_icbm152, 
    geo3d_snapped_ijk,
    meas_list
)

fluence_fname = os.path.join(pre_processed_path, "fluence_" + DATASET_NAME + ".h5")
sensitivity_fname = os.path.join(pre_processed_path, "sensitivity_" + DATASET_NAME + ".h5")

# compute fluence and sensitivity only once
# fwm.compute_fluence_mcx(fluence_fname)
# fwm.compute_sensitivity(fluence_fname, sensitivity_fname)

Adot = load_Adot(sensitivity_fname)

recon = None  # NEW: recon is now created per-recording/per-view in the main loop


Number of channels in rec["amp"]: 1220
Number of channels in meas_list: 1220


In [30]:
import cedalion.vis.blocks as vbx
import pyvista as pv
import cedalion.dataclasses as cdc
# keep only points that are not of type "landmark", i.e. source and detector points
geo3d_snapped_ijk = geo3d_snapped_ijk[geo3d_snapped_ijk.type != cdc.PointType.LANDMARK]

# now we plot the head same as before...
plt = pv.Plotter()
vbx.plot_surface(plt, head_icbm152.brain, color="#d3a6a1")
vbx.plot_surface(plt, head_icbm152.scalp, opacity=.1)
# but use the plot_labeled_points() function to add the snapped geo3d.
# The flag "show_labels" can be used to show the source, detector, and landmark names
vbx.plot_labeled_points(plt, geo3d_snapped_ijk, show_labels=True)
# plt.show()
# save image
plt.screenshot("head_with_snapped_points_" + DATASET_NAME + ".png")

pyvista_ndarray([[[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 ...,

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
  

In [31]:
import cedalion.vis.blocks as vbx
from cedalion.io.forward_model import FluenceFile

# pull fluence values from the corresponding source and detector pair

with FluenceFile(fluence_fname) as fluence_file:
    f = fluence_file.get_fluence("S12", 760) * fluence_file.get_fluence("D19", 760)

f = np.log10(np.clip(f, min=f[f > 0].min()))
vf = pv.wrap(f)

plt = pv.Plotter()

plt.add_volume(
    vf,
    log_scale=False,
    cmap="plasma_r",
    clim=(-10, -0),
    scalar_bar_args={
        "title": r"$log_{10}("
        r"F(\vec{x}_{src},\vec{x}) * F(\vec{x}, \vec{x}_{det})"
        ")$"
    },
)
vbx.plot_surface(plt, head_icbm152.brain, color="w")
vbx.plot_labeled_points(plt, geo3d_snapped_ijk, show_labels=False)
vbx.camera_at_cog(plt, head_icbm152.brain, rpos=[300, 150, 150])
# plt.show()
plt.screenshot(f"fluence_visualization_{DATASET_NAME}.png")

pyvista_ndarray([[[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 ...,

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
  

In [32]:
import cedalion.vis.anatomy.sensitivity_matrix as sensitivity_matrix

plotter = sensitivity_matrix.Main(
    sensitivity=Adot,
    brain_surface=head_icbm152.brain,
    head_surface=head_icbm152.scalp,
    labeled_points=geo3d_snapped_ijk,
)
plotter.plot(high_th=0, low_th=-3)
# plotter.plt.show()
# save the figure
plotter.plt.screenshot(f"sensitivity_visualization_{DATASET_NAME}.png")

pyvista_ndarray([[[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 ...,

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
  

In [33]:
pre_processed_path

PosixPath('datasets/pre_processed/imageRecon_params/BS_Laura/full')

In [17]:
# NEW: recon debug check updated for per-view creation flow
print(len(meas_list['channel']))
print(rec["amp"].channel.size)
if recon is None:
    print("recon is configured per recording/view in the main loop")
else:
    print(recon._F.shape)


2440
1220
recon is configured per recording/view in the main loop


In [18]:
stim = cedalion.io.read_events_from_tsv(files[0].replace('nirs.snirf', 'events.tsv'))

stim, rec = standardize_trial_types(DATASET_NAME, files[0], stim, rec)
rec.stim.head()

,onset,duration,value,trial_type
0,28.6013,5,1,left
1,50.9312,5,1,right
2,75.5938,5,1,right
3,97.5138,5,1,left
4,122.9975,5,1,left


In [19]:
print(np.diff(np.sort(rec.stim.onset.values)))
print(np.min(np.diff(np.sort(rec.stim.onset.values))))

[22.3299 24.6626 21.92   25.4837 23.7375 21.1512 25.8688 22.285  24.5713
 22.9162 25.3387 22.8688 21.4813]
21.15119999999999


In [20]:
# JOB-ARRAY HANDOFF: heavy image-recon preprocessing now runs outside the notebook.
# 1) Create the file list:
#    python src/subset/make_recon_param_job_list.py --dataset BS_Laura --subset full
#    This writes recon_param_files.txt inside pre_processed_path.
#
# 2) Submit the array job after checking the printed job_array range.
#    The %10 cap means at most 10 array tasks run at the same time:
#    sbatch \
#   --export=ALL,DATASET=vfc_hd,SUBSET=full \
#   --array=0-16%10 \
#   scripts/sbatch_recon_param_preprocessing.sh
#
# 3) After all jobs finish, run this cell to load per-recording metadata.

metadata_dir = pre_processed_path / "_job_metadata"
if not metadata_dir.exists():
    raise FileNotFoundError(
        f"Missing metadata directory: {metadata_dir}. Run the SLURM preprocessing jobs first."
    )

file_sens_drop_parcels_dict = {}
skipped_subjects = []
job_metadata = []

for meta_file in sorted(metadata_dir.glob("*.pkl")):
    with open(meta_file, "rb") as handle:
        meta = pickle.load(handle)
    job_metadata.append(meta)

    if meta.get("status") == "ok":
        file_sens_drop_parcels_dict[meta["file"]] = {
            "sensitive_parcels": meta["sensitive_parcels"],
            "dropped_parcels": meta["dropped_parcels"],
        }
    else:
        skipped_subjects.append(meta["file"])

with open(pre_processed_path / "file_sens_drop_parcels_dict.pkl", "wb") as handle:
    pickle.dump(file_sens_drop_parcels_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

expected_jobs = len(files)
finished_jobs = len(job_metadata)
print(f"metadata files loaded: {finished_jobs} / expected raw files: {expected_jobs}")
print(f"processed recordings: {len(file_sens_drop_parcels_dict)}")
print(f"skipped recordings: {len(skipped_subjects)}")

if finished_jobs != expected_jobs:
    missing = expected_jobs - finished_jobs
    print(f"WARNING: metadata count does not match expected raw files. Missing jobs: {missing}")

if skipped_subjects:
    print("Skipped files:")
    for f in skipped_subjects:
        print(f"  {f}")


metadata files loaded: 45 / expected raw files: 45
processed recordings: 45
skipped recordings: 0


In [21]:
# JOB-ARRAY HANDOFF: file_sens_drop_parcels_dict is saved in the metadata reload cell above.
print(pre_processed_path / "file_sens_drop_parcels_dict.pkl")



datasets/pre_processed/imageRecon_params/BS_Laura/full/file_sens_drop_parcels_dict.pkl


In [22]:
# Load the common parcel template used for segmentation.
if DATASET_NAME == "BS_Laura":
    template_dataset_name = "BallSqueezingHD_modified" 
else:
    template_dataset_name = DATASET_NAME
    
sens_parcel_template_path = Path(f"datasets/parcel_templates/parcel_template_{template_dataset_name}.pkl")

if not sens_parcel_template_path.exists():
    raise FileNotFoundError(f"Could not find parcel template: {sens_parcel_template_path}")

with open(sens_parcel_template_path, "rb") as handle:
    template_sens_parcel_list = pickle.load(handle)

print("template:", sens_parcel_template_path)
print("template parcels:", len(template_sens_parcel_list))

# For each processed recording, check how many sensitive parcels are contained in the common template.
for f, sens_drop_info in file_sens_drop_parcels_dict.items():
    sensitive_parcels = sens_drop_info["sensitive_parcels"]
    dropped_parcels = sens_drop_info["dropped_parcels"]

    num_sensitive_in_template = sum(1 for parcel in sensitive_parcels if parcel in template_sens_parcel_list)
    num_dropped_in_template = sum(1 for parcel in dropped_parcels if parcel in template_sens_parcel_list)

    print(f"  Sensitive parcels in template: {num_sensitive_in_template} / {len(template_sens_parcel_list)}")


template: datasets/parcel_templates/parcel_template_BallSqueezingHD_modified.pkl
template parcels: 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 / 110
  Sensitive parcels in template: 110 

In [25]:
Adot.shape

(1220, 35018, 2)

In [26]:
# import matplotlib.pyplot as plt
# # Plot as barplot the number of sensitive parcels in the template for each subject
# files = list(file_sens_drop_parcels_dict.keys())
# num_sensitive_in_template_list = []
# for f in files:
#     sensitive_parcels = file_sens_drop_parcels_dict[f]['sensitive_parcels']
#     num_sensitive_in_template = len(set(sensitive_parcels) & set(template_sens_parcel_list))
#     num_sensitive_in_template_list.append(num_sensitive_in_template)
# plt.figure(figsize=(10, 6))
# plt.bar(range(len(files)), num_sensitive_in_template_list, color='blue')
# # print the number of sensitive parcels in the template for each subject on top of the bars
# for i, num in enumerate(num_sensitive_in_template_list):    
#     plt.text(i, num + 0.5, str(num), ha='center', va='bottom')
# # plt.plot(range(len(files)), num_sensitive_in_template_list, color='red')
# plt.xticks(range(len(files)), [f.split('/')[3].replace('.snirf', '') for f in files], rotation=45)
# plt.xlabel('Subjects')
# plt.ylabel('Number of Sensitive Parcels in Template')
# plt.ylim(0, len(template_sens_parcel_list) + 10)
# plt.axhline(y=len(template_sens_parcel_list), color='gray', linestyle='dashed')
# # plt.legend()
# # plt.title('Sensitive Parcels in Template for Each Subject')
# plt.tight_layout()
# plt.show()

In [27]:
print("raw files:", len(files))
print("processed recordings from metadata:", len(file_sens_drop_parcels_dict))
print("skipped recordings:", len(skipped_subjects))


raw files: 45
processed recordings from metadata: 45
skipped recordings: 0


In [28]:
# JOB-ARRAY HANDOFF: rec["amp_clean"] is produced inside each external job and is not kept in notebook memory.
# Load saved .pkl files below for inspection instead.


In [29]:
# JOB-ARRAY HANDOFF: standardized event tables are stored in each saved .pkl as data["rec_stim"].
# Example: run the next data-loading cell, then inspect data["rec_stim"].


In [30]:
# JOB-ARRAY HANDOFF: use pre_processed_path-based glob patterns instead of hard-coded paths.


In [31]:
# NEW: load any augmented view for the first available preprocessed recording
pattern = str(pre_processed_path / "am_1__as_1" / "sub-*" / "*.pkl")
matched = sorted(glob.glob(pattern))
if not matched:
    raise FileNotFoundError(f"No files matched pattern: {pattern}")
with open(matched[0], "rb") as handle:
    data = pickle.load(handle)
print("loading:", matched[0])
data["rec_stim"].head()


loading: datasets/pre_processed/imageRecon_params/BS_Laura/full/am_1__as_1/sub-568/sub-568_task-BS_run-01_nirs.pkl


,onset,duration,value,trial_type
0,28.6013,10.0,1,left
1,50.9312,10.0,1,right
2,75.5938,10.0,1,right
3,97.5138,10.0,1,left
4,122.9975,10.0,1,left


In [32]:
pre_processed_path

PosixPath('datasets/pre_processed/imageRecon_params/BS_Laura/full')

In [33]:
# load data for visualization
if DATASET_NAME == "BallSqueezingHD_modified":
    subject_name = 'sub-185'
    base_name = f'{subject_name}_task-BallSqueezing_run-3_nirs'

elif DATASET_NAME == "FreshMotor":
    subject_name = 'sub-01'
    base_name = f'{subject_name}_task-FRESHMOTOR_run-left2s_nirs'

elif DATASET_NAME == "BS_Laura":
    subject_name = 'sub-583'
    subject_name = 'sub-633'
    base_name = f'{subject_name}_task-BS_run-01_nirs'
elif DATASET_NAME == "vfc_hd":
    subject_name = 'sub-12'
    base_name = f'{subject_name}_ses-01_task-WordStroop_run-01_nirs'
elif DATASET_NAME == "Anderson_sparse":
    subject_name = 'sub-1'
    base_name = f'{subject_name}_ses-1_task-WordStroop_run-1_nirs'

pattern = str(pre_processed_path / "am_1__as_1" / subject_name / f"{base_name}.pkl")
matched = sorted(glob.glob(pattern))
if not matched:
    raise FileNotFoundError(f"No files matched pattern: {pattern}")

file_to_plot = matched[0]
print("loading:", file_to_plot)

# load data
with open(file_to_plot, 'rb') as handle:
    data = pickle.load(handle)


loading: datasets/pre_processed/imageRecon_params/BS_Laura/full/am_1__as_1/sub-633/sub-633_task-BS_run-01_nirs.pkl


In [34]:
data['delta_conc']

<xarray.DataArray (time: 1386, parcel: 601, chromo: 2)> Size: 13MB
array([[[ 0.00000000e+00,  0.00000000e+00],
        [-1.54243096e-04, -2.58094109e-04],
        [-8.75469558e-05, -1.08578086e-04],
        ...,
        [ 7.62112445e-06,  6.45359071e-06],
        [ 4.67792487e-07, -1.19641923e-07],
        [ 8.88637583e-04,  6.03283677e-04]],

       [[ 0.00000000e+00,  0.00000000e+00],
        [-1.97615250e-04, -2.87395349e-04],
        [-1.06650482e-04, -1.18862019e-04],
        ...,
        [ 7.97875300e-06,  5.56887237e-06],
        [ 6.14994780e-07,  8.85551167e-08],
        [ 1.04626979e-03,  6.50314965e-04]],

       [[ 0.00000000e+00,  0.00000000e+00],
        [-2.23429276e-04, -3.05941890e-04],
        [-1.19409264e-04, -1.24865297e-04],
        ...,
...
        ...,
        [ 1.62927376e-05, -1.23473718e-06],
        [-1.43519114e-08,  8.34079455e-07],
        [ 1.67167526e-03, -2.37026768e-04]],

       [[ 0.00000000e+00,  0.00000000e+00],
        [ 3.01964413e-04,  1.04244853e-04],
        [ 1.46657972e-04,  5.89868498e-05],
        ...,
        [ 1.60926174e-05, -2.94593478e-06],
        [ 2.10940316e-07,  7.58638463e-07],
        [ 1.70349902e-03, -2.80392428e-04]],

       [[ 0.00000000e+00,  0.00000000e+00],
        [ 2.73108227e-04,  9.74927619e-05],
        [ 1.39607148e-04,  5.67567961e-05],
        ...,
        [ 1.36828127e-05, -5.94169844e-06],
        [ 3.82008190e-07,  6.70662832e-07],
        [ 1.59586488e-03, -4.04445405e-04]]], shape=(1386, 601, 2))
Coordinates:
  * chromo   (chromo) <U3 24B 'HbO' 'HbR'
  * time     (time) float64 11kB 33.99 34.23 34.46 34.69 ... 353.8 354.0 354.3
    samples  (time) int64 11kB 147 148 149 150 151 ... 1528 1529 1530 1531 1532
  * parcel   (parcel) object 5kB 'Background+FreeSurfer_Defined_Medial_Wall' ...

In [35]:
data.keys()

dict_keys(['conc_pcr', 'delta_conc', 'rec_stim', 'sensitive_parcels', 'view_id', 'alpha_meas', 'alpha_spatial', 'alpha_meas_0', 'alpha_spatial_0', 'alpha_meas_multiplier', 'alpha_spatial_multiplier', 'k_alpha_meas'])

<!-- # Calculate block averages in optical density
 -->


In [36]:
data['conc_pcr']

Magnitude,[[[0.0020587016723124726 -0.09912270733167491 0.18232375477385712 ... 0.010976412569953076 -0.05431020840805009 0.01579985335125543] [0.08983080161690953 0.08380430779091641 0.05489993830221662 ... -0.10983071668519212 -0.17924505175703903 -0.10691757563581555] [0.06836337371478574 0.030507794369594624 0.11071689799132128 ... -0.010010073520510782 -0.0024390633082090607 -0.04187149142234472] ... [-0.10364681379160288 -0.13036934935853983 -0.043653714304953364 ... -0.18595011037466486 -0.15492010872810894 -0.1541792794877846] [-0.0018220913371552248 0.5604459236748979 -0.26758306079953054 ... -0.00018075163740119936 0.00012155449316907443 0.0011238394968343833] [0.048717273907645804 -0.659733658157776 0.4567974694366916 ... -0.15887349347878127 -0.13319797157581986 -0.14621149465254651]] [[-0.0924844566588856 -0.19185852627312294 -0.06593650801542814 ... -0.025910183084995413 -0.023368126021301616 0.003117864523757549] [-0.15399477752442592 -0.3322458065199915 -0.03528710384168185 ... -0.0148310790177604 0.06286499298590296 0.0535638218446112] [-0.1411665392327361 -0.3168208832964636 -0.0705454198135256 ... -0.030992491794010677 -0.012213353837884793 0.029648333705777866] ... [-0.08934927010510378 -0.5806099305829354 0.27790221218195643 ... 0.17890619905552968 0.1440905205759362 0.07502069278221053] [0.0010661716794938452 -0.1817817972443504 0.09245220778953882 ... -0.01440024715862912 -0.016666578608783813 -0.009607434630971112] [-0.12444619354310149 -2.081952355353253 0.832420422014681 ... 0.025602399501626155 0.044867477884689075 0.06455900611324468]]]
Units,micromolar


In [37]:
# # Recompute od_pcr1 for one recording only, for visualization functions that need rec["od_pcr1"]

# from cedalion.io import read_events_from_tsv

# raw_file_for_viz = files[0]  # or set manually

# records = cedalion.io.read_snirf(raw_file_for_viz)
# rec = records[0]

# rec["amp"] = rec["amp"].sel(channel=subset_channels)

# stim = read_events_from_tsv(raw_file_for_viz.replace("nirs.snirf", "events.tsv"))
# rec.stim = rec.stim.sort_values(by="onset")
# stim, rec = standardize_trial_types(DATASET_NAME, raw_file_for_viz, stim, rec)

# rec["rep_amp"] = quality.repair_amp(rec["amp"], median_len=3, method="linear")
# rec["od_amp"], baseline = nirs.cw.int2od(rec["rep_amp"], return_baseline=True)

# rec["od_tddr"] = motion_correct.tddr(rec["od_amp"])
# rec["od_tddr_wavel"] = motion_correct.wavelet(rec["od_tddr"])

# rec["od_hpfilt"] = rec["od_tddr_wavel"].cd.freq_filter(
#     fmin=0.008,
#     fmax=0,
#     butter_order=4,
# )

# rec["amp_clean"] = cedalion.nirs.cw.od2int(rec["od_hpfilt"], baseline)

# ch_preproc = {
#     "sci_thresh": 0.5,
#     "psp_thresh": 0.1,
#     "window_len": 5 * units.s,
#     "dark_sat_thresh": [1e-3, 0.84],
#     "perc_time_clean": 0.5,
# }

# list_bad_ch = get_bad_ch_mask(rec["amp_clean"], ch_preproc)

# dpf = xr.DataArray(
#     [6, 6],
#     dims="wavelength",
#     coords={"wavelength": rec["amp"].wavelength},
# )

# rec["conc"] = cedalion.nirs.cw.od2conc(
#     rec["od_hpfilt"],
#     rec.geo3d,
#     dpf,
#     spectrum="prahl",
# )

# chromo_var = quality.measurement_variance(
#     rec["conc"],
#     list_bad_channels=list_bad_ch,
#     bad_rel_var=1e6,
#     calc_covariance=False,
# )

# rec["conc_pcr"], gb_comp_rem = physio.global_component_subtract(
#     rec["conc"],
#     ts_weights=1 / chromo_var,
#     k=0,
#     spatial_dim="channel",
#     spectral_dim="chromo",
# )

# rec["od_pcr1"] = cedalion.nirs.cw.conc2od(
#     rec["conc_pcr"],
#     rec.geo3d,
#     dpf,
#     spectrum="prahl",
# )

# print("ready:", raw_file_for_viz)
# print(rec["od_pcr1"])

<!-- Blockaverage od_pcr1 -->

In [38]:
# # segment data into epochs
# if DATASET_NAME in ["BallSqueezingHD_modified", "BS_Laura"]:
#     epochs = rec['od_pcr1'].cd.to_epochs(
#         rec.stim,  # stimulus dataframe
#         ["left", "right"],  # select fingertapping events, discard others
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
# elif DATASET_NAME == "FreshMotor":
#     epochs = rec['od_pcr1'].cd.to_epochs(
#         rec.stim,  # stimulus dataframe
#         ["right"],  # select left/right events
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
# elif DATASET_NAME in ["vfc_hd", "Anderson_sparse"]:
#     epochs = rec['od_pcr1'].cd.to_epochs(
#         rec.stim,  # stimulus dataframe
#         ["WordCongruent", "WordIncongruent"],
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
    
# # calculate baseline
# baseline = epochs.sel(reltime=(epochs.reltime < 0)).mean("reltime")

# # subtract baseline
# epochs_blcorrected = epochs - baseline

# # group trials by trial_type. For each group individually average the epoch dimension
# blockaverage_od_pcr1 = epochs_blcorrected.groupby("trial_type").mean("epoch")

In [39]:
from IPython.display import Image
def display_image(fname : str):
    display(Image(data=open(fname,'rb').read(), format='png'))

In [40]:
results_path_prefix = f'results/{subset_type}/{DATASET_NAME}/{subject_name}'
os.makedirs(results_path_prefix, exist_ok=True)

In [41]:
# # Plot block averages. Please ignore errors if the plot is too small in the HD case

# filename = f"results/{subset_type}/{DATASET_NAME}/blockaverage_channel_space_{subset_type}.png"

# # noPlts2 = int(np.ceil(np.sqrt(len(blockaverage_od_pcr1.channel))))
# # f,ax = plt.subplots(noPlts2,noPlts2, figsize=(12,10))
# # ax = ax.flatten()
# # for i_ch, ch in enumerate(blockaverage_od_pcr1.channel):
# #     for ls, trial_type in zip(["-", "--"], blockaverage_od_pcr1.trial_type):
# #         ax[i_ch].plot(blockaverage_od_pcr1.reltime, blockaverage_od_pcr1.sel(wavelength=760, trial_type=trial_type, channel=ch), "r", lw=2, ls=ls)
# #         ax[i_ch].plot(blockaverage_od_pcr1.reltime, blockaverage_od_pcr1.sel(wavelength=850, trial_type=trial_type, channel=ch), "b", lw=2, ls=ls)

# #     ax[i_ch].grid(1)
# #     ax[i_ch].set_title(ch.values)
# #     ax[i_ch].set_ylim(-.02, .02)
# #     ax[i_ch].set_axis_off()
# #     ax[i_ch].axhline(0, c="k")
# #     ax[i_ch].axvline(0, c="k")

# # # plt.suptitle("760nm: r | 850nm: b | left: - | right: --")
# # plt.suptitle("HbO: r | HbR: b | left: - | right: --")

# # plt.tight_layout()
# # plt.savefig(filename)

In [42]:
def match_landmark_labels(rec):
    subject_nasion_mask = rec.geo3d['label'].data == 'NASION'

    new_labels = rec.geo3d['label'].data.copy()
    new_labels[subject_nasion_mask] = 'Nz'

    # Create new geo3d with updated labels
    rec.geo3d = rec.geo3d.assign_coords(label=new_labels)
    # print(rec.geo3d['label'].data)

    return rec

In [43]:
# rec = match_landmark_labels(rec)

In [44]:
# # Viz reconstruction on Channel Space
# import cedalion.vis.anatomy
# filename_scalp = f"results/{subset_type}/{DATASET_NAME}/scalp_plot_ts_{subset_type}.png"

# data_ts = blockaverage_od_pcr1.sel(wavelength=850, trial_type="right")
# # data_ts = blockaverage_od_pcr1.sel(wavelength=850, trial_type="WordCongruent")
# # scalp_plot_gif expects the time dimension to be named 'time'
# data_ts = data_ts.rename({"reltime": "time"})

# # call plot function 
# cedalion.vis.anatomy.scalp_plot_gif(
#     data_ts,
#     rec.geo3d,
#     filename=filename_scalp,
#     time_range=(-5, 11, 0.5) * units.s,
#     scl=(-0.01, 0.01),
#     fps=6,
#     optode_size=6,
#     optode_labels=True,
#     str_title="OD 850 nm",
# )
# display_image(f"{filename_scalp}.gif")

Blockaverage_delta_conc

In [45]:
# if DATASET_NAME == "BallSqueezingHD_modified" or DATASET_NAME == "BS_Laura":
#     epochs = data['delta_conc'].cd.to_epochs(
#         data['rec_stim'],  # stimulus dataframe
#         ["left", "right"],  # select fingertapping events, discard others
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
# elif DATASET_NAME == "FreshMotor":
#     # segment data into epochs
#     epochs = data['delta_conc'].cd.to_epochs(
#         data['rec_stim'],  # stimulus dataframe
#         ["right"],  # select fingertapping events, discard others
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
    
# elif DATASET_NAME in ["vfc_hd", "Anderson_sparse"]:
#     # segment data into epochs
#     epochs = data['delta_conc'].cd.to_epochs(
#         data['rec_stim'],  # stimulus dataframe
#         ["WordCongruent", "WordIncongruent"],  # select fingertapping events, discard others
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )    

# # calculate baseline
# baseline = epochs.sel(reltime=(epochs.reltime < 0)).mean("reltime")

# # subtract baseline
# epochs_blcorrected = epochs - baseline

# # group trials by trial_type. For each group individually average the epoch dimension
# blockaverage_delta_conc = epochs_blcorrected.groupby("trial_type").mean("epoch")
# # blockaverage_delta_conc

In [46]:
# vertex_parcels = head_icbm152.brain.vertex_coords['parcel']
# vertex_parcels = np.array(vertex_parcels)

# parcel_index = blockaverage_delta_conc.get_index("parcel")  # pandas index
# vertex_parcel_idx = parcel_index.get_indexer(vertex_parcels)

# parcel_data = blockaverage_delta_conc.values  # shape (2, 107, 601, 2)

# # Broadcast using integer indexing on axis=2 (parcel axis)
# vertex_activity = parcel_data[:, :, vertex_parcel_idx, :]

# n_vertices = len(vertex_parcel_idx)

# vertex_da = xr.DataArray(
#     vertex_activity,
#     dims=("trial_type", "reltime", "vertex", "chromo"),
#     coords=dict(
#         trial_type=blockaverage_delta_conc.trial_type,
#         reltime=blockaverage_delta_conc.reltime,
#         chromo=blockaverage_delta_conc.chromo,

#         # vertex index
#         vertex=np.arange(n_vertices),

#         # parcel label of each vertex
#         parcel_of_vertex=("vertex", vertex_parcels),

#         # NEW: is_brain flag
#         is_brain=("vertex", np.ones(n_vertices, dtype=bool))
#     )
# )

# # vertex_da

In [47]:
subset_type

'full'

In [48]:
# filename_multiview = f'{results_path_prefix}/image_recon_multiview_{subset_type}'

# # prepare data
# # X_ts = vertex_da.sel(trial_type="right").rename({"reltime": "time"})
# X_ts = vertex_da.sel(trial_type="WordCongruent").rename({"reltime": "time"})
# X_ts = X_ts.transpose("vertex", "chromo", "time")

# # t_plot = 5.0
# # X_frame = X_ts.sel(time=t_plot, method="nearest")

# scl = np.percentile(np.abs(X_ts.sel(chromo='HbO')).pint.dequantify(), 99)
# clim = (-scl,scl)

# cedalion.vis.anatomy.image_recon_multi_view(
# # cedalion.vis.anatomy.image_recon_view(
#     X_ts,  # time series data; can be 2D (static) or 3D (dynamic)
#     # X_frame,
#     head_icbm152,
#     cmap='seismic',
#     clim=clim,
#     view_type='hbo_brain',
#     title_str='HbO / µM',
#     filename=filename_multiview,
#     SAVE=True,
#     time_range=(-2,10,0.5)*units.s,
#     fps=5,
#     geo3d_plot = None, #  geo3d_plot
#     wdw_size = (1024, 768)
# )
# display_image(filename_multiview+'.gif')

In [49]:
# import pickle
# from pathlib import Path

# import numpy as np
# import matplotlib.pyplot as plt
# import xarray as xr

# # Required variables from previous notebook cells:
# # subject_name
# # base_name
# # pre_processed_path
# # results_path_prefix
# # head_icbm152
# # get_param_config_folder

# alpha_meas_grid = [0.1, 1.0, 10.0]
# alpha_spatial_grid = [0.1, 1.0, 10.0]

# plot_chromo = "HbO"
# t_plot = 7.0  # absolute time in seconds from the saved delta_conc time axis

# out_dir = Path(results_path_prefix) / "image_recon_view_3x3_pngs"
# out_dir.mkdir(parents=True, exist_ok=True)

# vertex_parcels = np.asarray(head_icbm152.brain.vertex_coords["parcel"])
# brain_vertex = np.arange(len(vertex_parcels))

# png_files = {}
# grid_meta = {}
# all_vals = []

# # First pass: load all configs, map parcel data to brain vertices, collect color scale
# vertex_frames = {}

# for am_mult in alpha_meas_grid:
#     for as_mult in alpha_spatial_grid:
#         config_folder = get_param_config_folder(am_mult, as_mult)

#         file_to_load = (
#             pre_processed_path
#             / config_folder
#             / subject_name
#             / f"{base_name}.pkl"
#         )

#         if not file_to_load.exists():
#             raise FileNotFoundError(f"Missing file: {file_to_load}")

#         with open(file_to_load, "rb") as handle:
#             data_grid = pickle.load(handle)

#         delta = data_grid["delta_conc"]

#         # Select one time point, but keep both HbO and HbR because image_recon_view expects chromo.
#         snap = delta.sel(time=t_plot, method="nearest")

#         parcel_index = snap.get_index("parcel")
#         vertex_parcel_idx = parcel_index.get_indexer(vertex_parcels)

#         vertex_values = np.full(
#             (len(vertex_parcels), len(snap.chromo)),
#             np.nan,
#             dtype=float,
#         )

#         valid = vertex_parcel_idx >= 0
#         vertex_values[valid, :] = snap.values[vertex_parcel_idx[valid], :]

#         X_frame = xr.DataArray(
#             vertex_values,
#             dims=("vertex", "chromo"),
#             coords={
#                 "vertex": brain_vertex,
#                 "chromo": snap.chromo.values,
#                 "parcel_of_vertex": ("vertex", vertex_parcels),
#                 "is_brain": ("vertex", np.ones(len(vertex_parcels), dtype=bool)),
#             },
#         )

#         vertex_frames[(am_mult, as_mult)] = X_frame

#         vals = X_frame.sel(chromo=plot_chromo).values
#         all_vals.append(vals[np.isfinite(vals)])

#         grid_meta[(am_mult, as_mult)] = {
#             "alpha_meas": data_grid.get("alpha_meas"),
#             "alpha_spatial": data_grid.get("alpha_spatial"),
#         }

# all_vals = np.concatenate(all_vals)
# scl = np.percentile(np.abs(all_vals), 99)

# if not np.isfinite(scl) or scl == 0:
#     scl = 1.0

# clim = (-scl, scl)

# # Second pass: save one superior-view PNG per config using image_recon_view()
# for am_mult in alpha_meas_grid:
#     for as_mult in alpha_spatial_grid:
#         X_frame = vertex_frames[(am_mult, as_mult)]

#         png_base = out_dir / f"superior_am_{am_mult:g}__as_{as_mult:g}"

#         cedalion.vis.anatomy.image_recon_view(
#             X_frame,
#             head_icbm152,
#             cmap="seismic",
#             clim=clim,
#             view_type="hbo_brain",
#             view_position="superior",
#             title_str=f"{plot_chromo} / uM",
#             filename=str(png_base),
#             SAVE=True,
#             geo3d_plot=None,
#             wdw_size=(700, 700),
#         )

#         png_files[(am_mult, as_mult)] = Path(str(png_base) + ".png")

# # Third pass: compose the 9 saved PNGs into one matplotlib figure
# fig, axes = plt.subplots(3, 3, figsize=(12, 12))

# for row, am_mult in enumerate(alpha_meas_grid):
#     for col, as_mult in enumerate(alpha_spatial_grid):
#         ax = axes[row, col]
#         png_file = png_files[(am_mult, as_mult)]
#         meta = grid_meta[(am_mult, as_mult)]

#         img = plt.imread(png_file)
#         ax.imshow(img)
#         ax.axis("off")

#         ax.set_title(
#             f"am x{am_mult:g}, as x{as_mult:g}\n"
#             f"am={meta['alpha_meas']:.2e}, as={meta['alpha_spatial']:.2e}",
#             fontsize=8,
#         )

# fig.suptitle(
#     f"Superior view using image_recon_view | "
#     f"{plot_chromo} @ t={float(t_plot):.2f}s | "
#     f"{subject_name} | {base_name}",
#     fontsize=11,
# )

# fig.tight_layout()

# combined_png = Path(results_path_prefix) / f"image_recon_view_superior_3x3_t{t_plot:g}.png"
# fig.savefig(combined_png, dpi=200, bbox_inches="tight")

# plt.show()

# print("Saved combined figure:", combined_png)
# print("Saved individual PNGs in:", out_dir)

# Segmentation

In [50]:
pre_processed_path

PosixPath('datasets/pre_processed/imageRecon_params/BS_Laura/full')

In [51]:
# load (all) parcel files
preproc_files_path = str(pre_processed_path / 'am_*__as_*' / 'sub-*' / '*.pkl')
proc_pkl_files = glob.glob(preproc_files_path)

len(proc_pkl_files), proc_pkl_files[:2]

(405,
 ['datasets/pre_processed/imageRecon_params/BS_Laura/full/am_1__as_0.1/sub-583/sub-583_task-BS_run-02_nirs.pkl',
  'datasets/pre_processed/imageRecon_params/BS_Laura/full/am_1__as_0.1/sub-583/sub-583_task-BS_run-01_nirs.pkl'])

In [52]:
processed_path = Path(f'datasets/processed/{augmentation_strategy}/{DATASET_NAME}/{subset_type}')
processed_path.mkdir(parents=True, exist_ok=True)
processed_path

use_channel_space = True

### Create template sensitive parcels

In [53]:
create_orload_parcel_template = "load"  # "load" or "create"

if create_orload_parcel_template == "create":
    # Load any file as all processed files have the same sensitive parcels
    # f = f"datasets/{subset_type}_pre_processed/BallSqueezingHD_modified/sub-185/sub-185_task-BallSqueezing_run-1_nirs.pkl"
    
    if DATASET_NAME == "BallSqueezingHD_modified":
        f = sorted(glob.glob("datasets/pre_processed/BallSqueezingHD_modified/*/sub-185/sub-185_task-BallSqueezing_run-1_nirs.pkl"))[0]
    elif DATASET_NAME == "vfc_hd":
        # f = sorted(glob.glob("datasets/pre_processed/vfc_hd/*/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs.pkl"))[0]
        f = sorted(glob.glob("datasets/pre_processed/imageRecon_params/vfc_hd/full/am_1__as_1/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs.pkl"))[0]
    elif DATASET_NAME == "Anderson_sparse":
        f = sorted(glob.glob("datasets/pre_processed/Anderson_sparse/*/sub-1/sub-1_ses-1_task-WordStroop_run-1_nirs.pkl"))[0]
        
    with open(f, 'rb') as handle:
        data = pickle.load(handle)
        
    print(len(data['sensitive_parcels']))

    # save the loaded sensitive parcels as template as pkl file
    folder_path = 'datasets/parcel_templates'
    os.makedirs(folder_path, exist_ok=True)
    # sens_parcel_template_path = os.path.join(folder_path, f'parcel_template_{DATASET_NAME}.pkl')
    # sens_parcel_template_path = os.path.join(folder_path, f'{subset_type}_parcel_template_subset_2_BallSqueezingHD_modified.pkl')
    sens_parcel_template_path = "datasets/parcel_templates/parcel_template_vfc_hd.pkl"
    # sens_parcel_template_path = "datasets/parcel_templates/parcel_template_Anderson_sparse.pkl"

    with open(sens_parcel_template_path, 'wb') as handle:
        pickle.dump(data['sensitive_parcels'], handle)
    
    print(f"Parcel template saved to {sens_parcel_template_path}")
        
        
elif create_orload_parcel_template == "load":
    folder_path = 'datasets/parcel_templates'
    
    if DATASET_NAME == "BallSqueezingHD_modified" or DATASET_NAME == "BS_Laura":
        # sens_parcel_template_path = os.path.join(folder_path, 'subset_2_parcel_template_subset_2_BallSqueezingHD_modified.pkl')
        sens_parcel_template_path = os.path.join(folder_path, 'parcel_template_BallSqueezingHD_modified.pkl')
    elif DATASET_NAME == "vfc_hd" or DATASET_NAME == "Anderson_sparse":
        sens_parcel_template_path = os.path.join(folder_path, 'parcel_template_vfc_hd.pkl')
        # sens_parcel_template_path = os.path.join(folder_path, 'parcel_template_Anderson_sparse.pkl')
    
    with open(sens_parcel_template_path, 'rb') as handle:
        template_sens_parcel_list = pickle.load(handle)
        
    print(len(template_sens_parcel_list))
    print(f"Parcel template loaded from {sens_parcel_template_path}")

110
Parcel template loaded from datasets/parcel_templates/parcel_template_BallSqueezingHD_modified.pkl


In [54]:
# check how many of the sensitive parcels are motor are
motor_labels = [l for l in template_sens_parcel_list if l.startswith("SomMot")]
print(f"{len(motor_labels)} motor labels")

47 motor labels


In [55]:
template_sens_parcel_list

['ContA_IPS_2_LH',
 'ContA_IPS_2_RH',
 'ContA_IPS_3_LH',
 'ContA_IPS_3_RH',
 'ContA_PFCd_1_LH',
 'ContA_PFCd_2_LH',
 'ContA_PFCl_1_LH',
 'ContA_PFCl_2_LH',
 'ContA_PFCl_2_RH',
 'ContA_PFCl_5_RH',
 'ContA_PFCl_6_LH',
 'ContB_IPL_3_LH',
 'ContB_IPL_3_RH',
 'ContB_PFCd_1_LH',
 'ContB_PFCl_1_LH',
 'ContB_PFCld_2_RH',
 'ContB_PFCld_3_RH',
 'ContB_PFCld_4_RH',
 'ContB_PFCld_5_RH',
 'ContB_PFCld_6_RH',
 'DefaultB_IPL_3_LH',
 'DefaultB_PFCl_1_LH',
 'DefaultB_PFCl_2_LH',
 'DefaultB_Temp_2_RH',
 'DefaultB_Temp_5_LH',
 'DorsAttnA_SPL_7_RH',
 'DorsAttnB_FEF_1_LH',
 'DorsAttnB_FEF_1_RH',
 'DorsAttnB_FEF_2_RH',
 'DorsAttnB_FEF_4_LH',
 'DorsAttnB_FEF_4_RH',
 'DorsAttnB_PostC_1_LH',
 'DorsAttnB_PostC_1_RH',
 'DorsAttnB_PostC_2_RH',
 'DorsAttnB_PostC_3_LH',
 'DorsAttnB_PostC_3_RH',
 'DorsAttnB_PostC_4_RH',
 'DorsAttnB_PostC_9_RH',
 'DorsAttnB_PrCv_1_LH',
 'DorsAttnB_PrCv_1_RH',
 'SalVentAttnA_FrOper_3_LH',
 'SalVentAttnA_FrOper_3_RH',
 'SalVentAttnA_FrOper_5_RH',
 'SalVentAttnA_ParOper_1_RH',
 'SalVent

In [56]:
min_len_sens_parcels = float('inf')
min_len_i = None

for i in range(len(proc_pkl_files)):
    with open(proc_pkl_files[i], 'rb') as handle:
        data_pickle = pickle.load(handle)
        delta_brain = data_pickle['delta_conc']
        sensitive_parcels = data_pickle['sensitive_parcels']
        print(f"File {i}: {len(sensitive_parcels)} sensitive parcels")
        if len(sensitive_parcels) < min_len_sens_parcels:
            min_len_sens_parcels = len(sensitive_parcels)
            min_len_i = i
            
        # also check how many of the sensitive parcels are motor are
        motor_labels = [l for l in sensitive_parcels if l.startswith("SomMot")]
        print(f"File {i}: {len(motor_labels)} motor labels")
        print("-"*50)
            
min_len_i, min_len_sens_parcels

File 0: 353 sensitive parcels
File 0: 70 motor labels
--------------------------------------------------
File 1: 353 sensitive parcels
File 1: 70 motor labels
--------------------------------------------------
File 2: 353 sensitive parcels
File 2: 70 motor labels
--------------------------------------------------
File 3: 353 sensitive parcels
File 3: 70 motor labels
--------------------------------------------------
File 4: 353 sensitive parcels
File 4: 70 motor labels
--------------------------------------------------
File 5: 353 sensitive parcels
File 5: 70 motor labels
--------------------------------------------------
File 6: 353 sensitive parcels
File 6: 70 motor labels
--------------------------------------------------
File 7: 353 sensitive parcels
File 7: 70 motor labels
--------------------------------------------------
File 8: 353 sensitive parcels
File 8: 70 motor labels
--------------------------------------------------
File 9: 353 sensitive parcels
File 9: 70 motor labels
-

(0, 353)

In [57]:
with open(proc_pkl_files[7], 'rb') as handle:
    data_pickle = pickle.load(handle)
    delta_brain = data_pickle['delta_conc']
    sensitive_parcels = data_pickle['sensitive_parcels']
len(sensitive_parcels)

353

In [58]:
from collections import Counter

# Gather all fs_mean values from all files
fs_list = []
for file in proc_pkl_files:
    with open(file, 'rb') as handle:
        data_pickle = pickle.load(handle)
    delta_brain = data_pickle['delta_conc']
    
    dt = np.diff(delta_brain.time.data)
    fs_mean = float(1.0 / np.mean(dt))
    
    fs_list.append(fs_mean)

fs_counts = Counter(fs_list)
len (proc_pkl_files), fs_counts

(405,
 Counter({4.324324324324325: 189,
          4.3243243243243255: 153,
          4.324324324324324: 63}))

In [59]:
with open(file, 'rb') as handle:
    data_pickle = pickle.load(handle)

delta_brain = data_pickle['delta_conc']
sensitive_parcels = data_pickle['sensitive_parcels']
rec_stim = data_pickle['rec_stim']

delta_brain.time.data

array([ 28.90625,  29.1375 ,  29.36875, ..., 348.95625, 349.1875 ,
       349.41875], shape=(1387,))

In [60]:
delta_brain.sel(parcel=template_sens_parcel_list).shape

(1387, 110, 2)

In [61]:
processed_path

PosixPath('datasets/processed/imageRecon_params/BS_Laura/full')

In [62]:
if not use_channel_space:
    baseline_duration = 2.5  # in seconds
    n_shifts = 9
    duration = 10  # in seconds
    post_padding = 5  # in seconds
    extract_rest_segments = True # for vfc_hd, Anderson_sparse
    resample = False
    delta_range = (-2.5, 2.5)
        
    if DATASET_NAME == "BallSqueezingHD_modified":
        n_timepoints = 87 # 244 # 174 # fixed length after shifting
    elif DATASET_NAME == "BS_Laura":
        n_timepoints = 87 # 244 # 174 # fixed length after shifting
        resample = True
    elif DATASET_NAME == "vfc_hd":
        n_timepoints = 174 # fixed length after shifting

    if DATASET_NAME == "FreshMotor":
        delta_range = (-2.0, 0.0)

    start_shift = np.linspace(*delta_range, n_shifts)
        
    for file in proc_pkl_files:
        with open(file, 'rb') as handle:
            data_pickle = pickle.load(handle)
        
        delta_brain = data_pickle['delta_conc']
        sensitive_parcels = data_pickle['sensitive_parcels']
        rec_stim = data_pickle['rec_stim']

        # [OLD]: Align subject-specific parcels to a common parcel template (zero-pad missing parcels)
        # delta_brain = delta_brain.sel(parcel=sensitive_parcels).reindex(parcel=PARCEL_TEMPLATE, fill_value=0)
        
        # [NEW]: select only parcels in the common template
        delta_brain = delta_brain.sel(parcel=template_sens_parcel_list)
        
        # ---- NEW: Resampling ---- use for Laura, (and ??)
        if resample:
            if DATASET_NAME == "BS_Laura":
                target_fs = 8.7 # 24.4 # 8.98876404494382  # From BSQ-HD
            dt = 1.0 / target_fs
            t0 = float(delta_brain.time.min())
            t1 = float(delta_brain.time.max())
            new_time = np.arange(t0, t1 + 1e-9, dt)
            delta_brain = delta_brain.interp(time=new_time)
        # -------------------------
        
        # Process event segments
        i = 0
        for index, row in rec_stim.iterrows():
            # Binary labeling: pool word conditions into "task" label
            trial_type_clean = row["trial_type"].lower()

            if "word" in trial_type_clean:
                label = f"{trial_type_clean}_task"
            else:
                label = trial_type_clean
                
            for s in start_shift:
                start_time = row["onset"] + s
                end_time = start_time + duration + post_padding # in seconds
                baseline = delta_brain.sel(
                    time=slice(row["onset"] - baseline_duration, row["onset"])
                ).mean("time")
                
                # Then, trimming is easy with `.sel()`:
                x = delta_brain.sel(time=slice(start_time, end_time)) - baseline
                x = x.isel(time=slice(0, n_timepoints))
                
                # new check
                # print(x.sizes["time"], " timepoints")
                if x.sizes["time"] < n_timepoints:
                    print(f"Skipping segment for file {os.path.basename(file)} at trial {i} due to insufficient length: {x.sizes['time']} < {n_timepoints}")
                    # continue  # skip short segment
                
                x = x.transpose("parcel", "chromo", "time")
                del x.time.attrs['units']

                if not os.path.exists(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path)))):
                    os.makedirs(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path))))
                if s == 0:
                    x.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+"_test.nc"))
                    i += 1
                else:
                    x.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+".nc"))
                    i += 1

        if extract_rest_segments:
            # --- REST-STATE SEGMENT EXTRACTION ---
            # Rest segments: start at event_onset + 15s, end at next_event_onset - 5s
            # Extracted with 0.5s sliding window, 10s duration each
            recording_start = float(delta_brain.time.values[0])
            recording_end = float(delta_brain.time.values[-1])
            segment_length_sec = duration  # 10 seconds
            rest_label = "rest"
            step_sec = 0.25  # sliding window step
            rest_segment_count = 0
            
            # Process rest intervals between consecutive events
            for idx in range(len(rec_stim)):
                current_onset = rec_stim.iloc[idx]["onset"]
                
                # Rest interval: [event_onset + 15s, next_event_onset - 5s]
                rest_start = current_onset + 15.0
                
                if idx + 1 < len(rec_stim):
                    next_onset = rec_stim.iloc[idx + 1]["onset"]
                    rest_end = next_onset - 5.0
                else:
                    rest_end = recording_end
                
                interval_length = rest_end - rest_start
                
                # Check if interval is long enough for at least one segment
                if interval_length < segment_length_sec:
                    print(f"  Skipping rest interval {idx}: [{rest_start:.1f}s - {rest_end:.1f}s] (length: {interval_length:.1f}s < {segment_length_sec}s)")
                    continue
                
                print(f"  Rest interval {idx}: [{rest_start:.1f}s - {rest_end:.1f}s] (length: {interval_length:.1f}s)")
                
                # First pass: collect valid rest segment starts in this interval
                t = rest_start
                valid_segment_starts = []
                
                while t + segment_length_sec <= rest_end:
                    baseline_start = max(t - baseline_duration, recording_start)
                    baseline_rest = delta_brain.sel(time=slice(baseline_start, t)).mean("time")
                    x_rest = delta_brain.sel(time=slice(t, t + segment_length_sec)) - baseline_rest
                    x_rest = x_rest.isel(time=slice(0, n_timepoints))
                    
                    # Skip if segment is too short
                    if x_rest.sizes["time"] < n_timepoints:
                        print(f"    Skipping rest segment at t={t:.1f}s (insufficient samples: {x_rest.sizes['time']} < {n_timepoints})")
                        t += step_sec
                        continue
                    
                    valid_segment_starts.append(t)
                    t += step_sec
                
                if len(valid_segment_starts) == 0:
                    continue
                
                # Select the segment centered closest to interval midpoint as test
                interval_midpoint = (rest_start + rest_end) / 2.0
                segment_centers = np.array(valid_segment_starts) + (segment_length_sec / 2.0)
                test_segment_idx = int(np.argmin(np.abs(segment_centers - interval_midpoint)))
                
                interval_segments = 0
                for seg_idx, t_start in enumerate(valid_segment_starts):
                    baseline_start = max(t_start - baseline_duration, recording_start)
                    baseline_rest = delta_brain.sel(time=slice(baseline_start, t_start)).mean("time")
                    x_rest = delta_brain.sel(time=slice(t_start, t_start + segment_length_sec)) - baseline_rest
                    x_rest = x_rest.isel(time=slice(0, n_timepoints))
                    x_rest = x_rest.transpose("parcel", "chromo", "time")
                    del x_rest.time.attrs['units']
                    
                    is_test_segment = seg_idx == test_segment_idx
                    rest_suffix = "_test.nc" if is_test_segment else ".nc"
                    rest_filename = file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", f"_{rest_label}_{rest_segment_count}{rest_suffix}")
                    x_rest.to_netcdf(rest_filename)
                    rest_segment_count += 1
                    interval_segments += 1
                
                print(f"  Extracted {interval_segments} rest segments from interval {idx} (middle segment index {test_segment_idx} saved as _test.nc)")
        
            print(f"Total rest segments extracted: {rest_segment_count}")

        print("finished processing file: ", os.path.basename(file).replace(".pkl",".npy"))

    print('\n--- Done!---')

In [65]:
if DATASET_NAME in ["vfc_hd", "Anderson_sparse"]:
    from collections import defaultdict
    segmented_files = glob.glob(str(processed_path / 'am_*__as_*' / 'sub-*' / '*.nc'))

    subject_label_counts = defaultdict(lambda: defaultdict(int))

    for file_path in segmented_files:
        filename = os.path.basename(file_path)
        subject = os.path.basename(os.path.dirname(file_path))
        
        if '_task_' in filename:
            label = 'task'
        elif '_rest_' in filename:
            label = 'rest'
        else:
            label = 'unknown'
        
        subject_label_counts[subject][label] += 1

    data_for_df = []
    for subject in sorted(subject_label_counts.keys()):
        counts = subject_label_counts[subject]
        data_for_df.append({
            'Subject': subject,
            'Task': counts.get('task', 0),
            'Rest': counts.get('rest', 0),
            'Total': counts.get('task', 0) + counts.get('rest', 0)
        })

    df_counts = pd.DataFrame(data_for_df)

    print(df_counts.to_string(index=False))

    print("="*40)
    print(f"Total subjects: {len(df_counts)}")
    print(f"Total task segments: {df_counts['Task'].sum()}")
    print(f"Total rest segments: {df_counts['Rest'].sum()}")
    print(f"Total segments: {df_counts['Total'].sum()}")
    print(f"\nClass balance: Task={df_counts['Task'].sum()}, Rest={df_counts['Rest'].sum()}")
    print(f"Task/Rest ratio: {df_counts['Task'].sum() / df_counts['Rest'].sum():.2f}" if df_counts['Rest'].sum() > 0 else "Task/Rest ratio: N/A")

In [66]:
if DATASET_NAME in ["vfc_hd", "Anderson_sparse"]:
    test_files = sorted(processed_path.rglob("*_test.nc"))

    counts = {"task": 0, "rest": 0}
    for f in test_files:
        name = f.name.lower()
        if "_rest_" in name:
            counts["rest"] += 1
        else:
            counts["task"] += 1

    df_test_counts = pd.DataFrame([
        {"Label": "task", "_test_files": counts["task"]},
        {"Label": "rest", "_test_files": counts["rest"]},
    ])

    display(df_test_counts)
    print(f"Total _test files: {len(test_files)}")


<!-- # Channel space segmentation -->

In [68]:
if use_channel_space:
    # Channel-space segmentation mirrors the parcel-space timing and labeling logic.
    # Channel data do not depend on image-reconstruction parameters, so only the
    # baseline am_1__as_1 copy is segmented.
    baseline_duration = 2.5  # seconds
    n_shifts = 9
    duration = 10  # seconds
    post_padding = 5  # seconds; candidate window is trimmed to n_timepoints
    extract_rest_segments = DATASET_NAME in ["vfc_hd", "Anderson_sparse"]

    channel_config = {
        "BallSqueezingHD_modified": {
            "n_timepoints": 87,
            "resample": False,
            "target_fs": None
        },
        "BS_Laura": {
            "n_timepoints": 87,
            "resample": True,
            "target_fs": 8.7
        },
        "FreshMotor": {
            "n_timepoints": 87,
            "resample": False,
            "target_fs": None
        },
        "vfc_hd": {
            "n_timepoints": 174,
            "resample": False,
            "target_fs": None
        },
        "Anderson_sparse": {
            "n_timepoints": 87,
            "resample": False,
            "target_fs": None
        },
    }

    config = channel_config[DATASET_NAME]
    n_timepoints = config["n_timepoints"]
    resample = config["resample"]
    target_fs = config["target_fs"]
    delta_range = (-2.0, 0.0) if DATASET_NAME == "FreshMotor" else (-2.5, 2.5)
    start_shift = np.linspace(*delta_range, n_shifts)

    # Keep channel-space outputs separate from parcel-space/image-recon outputs.
    channel_processed_path = Path(
        f"datasets/processed/channel_space/{DATASET_NAME}/{subset_type}"
    )
    channel_processed_path.mkdir(parents=True, exist_ok=True)

    baseline_config_path = pre_processed_path / "am_1__as_1"
    channel_proc_pkl_files = sorted(
        glob.glob(str(baseline_config_path / "sub-*" / "*.pkl"))
    )
    if not channel_proc_pkl_files:
        raise FileNotFoundError(
            f"No channel-space input files found under {baseline_config_path}"
        )

    print("channel-space input config:", baseline_config_path)
    print("channel-space output path:", channel_processed_path)
    print("channel-space recordings:", len(channel_proc_pkl_files))

    with open(channel_proc_pkl_files[0], "rb") as handle:
        first_channel_data = pickle.load(handle)["conc_pcr"]
    expected_channels = first_channel_data.channel.values.tolist()
    print("expected channel count:", len(expected_channels))

    for file in channel_proc_pkl_files:
        with open(file, "rb") as handle:
            data_pickle = pickle.load(handle)

        rec_stim = data_pickle["rec_stim"]
        conc_pcr = data_pickle["conc_pcr"]

        current_channels = conc_pcr.channel.values.tolist()
        missing_channels = sorted(set(expected_channels) - set(current_channels))
        extra_channels = sorted(set(current_channels) - set(expected_channels))
        if missing_channels or extra_channels:
            raise ValueError(
                f"Channel mismatch for {file}: "
                f"missing={missing_channels}, extra={extra_channels}"
            )
        conc_pcr = conc_pcr.sel(channel=expected_channels)

        if resample:
            dt = 1.0 / target_fs
            t0 = float(conc_pcr.time.min())
            t1 = float(conc_pcr.time.max())
            new_time = np.arange(t0, t1 + 1e-9, dt)
            conc_pcr = conc_pcr.interp(time=new_time)

        relative_file = Path(file).relative_to(baseline_config_path)
        output_base = channel_processed_path / relative_file
        output_base.parent.mkdir(parents=True, exist_ok=True)

        # Task segments: preserve the word condition in the filename while pooling
        # WordCongruent and WordIncongruent into the task class.
        segment_count = 0
        for _, row in rec_stim.iterrows():
            trial_type_clean = row["trial_type"].lower()
            
            label = (
                f"{trial_type_clean}_task"
                if "word" in trial_type_clean
                else trial_type_clean
            )

            baseline_conc_pcr = conc_pcr.sel(
                time=slice(
                    row["onset"] - baseline_duration,
                    row["onset"],
                )
            ).mean("time")

            for shift in start_shift:
                start_time = row["onset"] + shift
                end_time = start_time + duration + post_padding
                
                x_channel = (
                    conc_pcr.sel(time=slice(start_time, end_time))
                    - baseline_conc_pcr
                )
                x_channel = x_channel.isel(time=slice(0, n_timepoints))

                if x_channel.sizes["time"] < n_timepoints:
                    print(
                        f"Skipping channel task segment for {Path(file).name} "
                        f"at segment {segment_count}: "
                        f"{x_channel.sizes['time']} < {n_timepoints}"
                    )
                    segment_count += 1
                    continue

                x_channel = x_channel.transpose("channel", "chromo", "time")
                x_channel.time.attrs.pop("units", None)

                test_suffix = "_test.nc" if np.isclose(shift, 0.0) else ".nc"
                output_file = output_base.with_name(
                    f"{output_base.stem}_{label}_{segment_count}{test_suffix}"
                )
                x_channel.to_netcdf(output_file)
                segment_count += 1

        if extract_rest_segments:
            # Match the parcel-space rest definition for a fair comparison.
            recording_start = float(conc_pcr.time.values[0])
            recording_end = float(conc_pcr.time.values[-1])
            rest_step_seconds = 0.25
            rest_segment_count = 0

            for idx in range(len(rec_stim)):
                current_onset = rec_stim.iloc[idx]["onset"]
                rest_start = current_onset + 15.0
                
                if idx + 1 < len(rec_stim):
                    rest_end = rec_stim.iloc[idx + 1]["onset"] - 5.0
                else:
                    rest_end = recording_end

                if rest_end - rest_start < duration:
                    continue

                valid_starts = []
                t_start = rest_start
                while t_start + duration <= rest_end:
                    baseline_start = max(
                        t_start - baseline_duration,
                        recording_start,
                    )
                    baseline_rest = conc_pcr.sel(
                        time=slice(baseline_start, t_start)
                    ).mean("time")
                    x_rest = (
                        conc_pcr.sel(time=slice(t_start, t_start + duration))
                        - baseline_rest
                    )
                    x_rest = x_rest.isel(time=slice(0, n_timepoints))
                    if x_rest.sizes["time"] == n_timepoints:
                        valid_starts.append(t_start)
                    t_start += rest_step_seconds

                if not valid_starts:
                    continue

                interval_midpoint = (rest_start + rest_end) / 2.0
                segment_centers = np.asarray(valid_starts) + duration / 2.0
                test_idx = int(
                    np.argmin(np.abs(segment_centers - interval_midpoint))
                )

                for segment_idx, t_start in enumerate(valid_starts):
                    baseline_start = max(
                        t_start - baseline_duration,
                        recording_start,
                    )
                    baseline_rest = conc_pcr.sel(
                        time=slice(baseline_start, t_start)
                    ).mean("time")
                    x_rest = (
                        conc_pcr.sel(time=slice(t_start, t_start + duration))
                        - baseline_rest
                    )
                    x_rest = x_rest.isel(time=slice(0, n_timepoints))
                    x_rest = x_rest.transpose("channel", "chromo", "time")
                    x_rest.time.attrs.pop("units", None)

                    test_suffix = "_test.nc" if segment_idx == test_idx else ".nc"
                    output_file = output_base.with_name(
                        f"{output_base.stem}_rest_{rest_segment_count}{test_suffix}"
                    )
                    x_rest.to_netcdf(output_file)
                    rest_segment_count += 1

            print(f"rest segments saved for {Path(file).name}: {rest_segment_count}")

        print("finished channel processing:", Path(file).name)

    print("\n--- Channel-space segmentation done ---")


channel-space input config: datasets/pre_processed/imageRecon_params/BS_Laura/full/am_1__as_1
channel-space output path: datasets/processed/channel_space/BS_Laura/full
channel-space recordings: 45
expected channel count: 1220
finished channel processing: sub-568_task-BS_run-01_nirs.pkl
finished channel processing: sub-568_task-BS_run-02_nirs.pkl
finished channel processing: sub-568_task-BS_run-03_nirs.pkl
finished channel processing: sub-577_task-BS_run-01_nirs.pkl
finished channel processing: sub-577_task-BS_run-02_nirs.pkl
finished channel processing: sub-577_task-BS_run-03_nirs.pkl
finished channel processing: sub-580_task-BS_run-01_nirs.pkl
finished channel processing: sub-580_task-BS_run-02_nirs.pkl
finished channel processing: sub-580_task-BS_run-03_nirs.pkl
finished channel processing: sub-581_task-BS_run-01_nirs.pkl
finished channel processing: sub-581_task-BS_run-02_nirs.pkl
finished channel processing: sub-581_task-BS_run-03_nirs.pkl
finished channel processing: sub-583_task-

In [34]:
print("VFC data")

file_to_load_channel_vfc = "/home/orabe/fNIRS_sparseToDense/datasets/processed/channel_space/vfc_hd/full/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs_rest_0.nc"
data = xr.load_dataarray(file_to_load_channel_vfc)
print(f"channel: {data.shape[0]}, chromo: {data.shape[1]}, time: {data.shape[2]}")  # channel, chromo, time

file_to_load_parcel_vfc = "datasets/processed/imageRecon_params/vfc_hd/full/am_0.1__as_0.1/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs_rest_0.nc"
data = xr.load_dataarray(file_to_load_parcel_vfc)
print(f"parcel: {data.shape[0]}, chromo: {data.shape[1]}, time: {data.shape[2]}")  # parcel, chromo, time

print("\n" + "="*35 + "\n")
print("BS_Laura data")


file_to_load_channel_laura = "datasets/processed/channel_space/BS_Laura/full/sub-568/sub-568_task-BS_run-01_nirs_left_0.nc"
data = xr.load_dataarray(file_to_load_channel_laura)
print(f"channel: {data.shape[0]}, chromo: {data.shape[1]}, time: {data.shape[2]}")  # channel, chromo, time

file_to_load_parcel_laura = "datasets/processed/imageRecon_params/BS_Laura/full/am_0.1__as_0.1/sub-568/sub-568_task-BS_run-01_nirs_left_0.nc"
data = xr.load_dataarray(file_to_load_parcel_laura)
print(f"parcel: {data.shape[0]}, chromo: {data.shape[1]}, time: {data.shape[2]}")  # parcel, chromo, time

# print("\n Laura subset 2:")

# file_to_load_parcel_laura_subset_2 = "datasets_14062026/subset_2_processed/BallSqueezingHD_modified/sub-170/sub-170_task-BallSqueezing_run-2_nirs_left_18.nc"
# data = xr.load_dataarray(file_to_load_parcel_laura_subset_2)
# print(f"parcel: {data.shape[0]}, chromo: {data.shape[1]}, time: {data.shape[2]}")  # parcel, chromo, time

VFC data
channel: 214, chromo: 2, time: 174
parcel: 143, chromo: 2, time: 174


BS_Laura data
channel: 1220, chromo: 2, time: 87
parcel: 110, chromo: 2, time: 87


<!-- ### Old appraoch using event files and freq0.5 -->

<!-- FreshMotor

- parcel space: (parcel: 110, chromo: 2, time: 87)
- channel space: (time: 87, channel: 68, chromo: 2)

---
BallSqueezing

- parcel space: (parcel: 110, chromo: 2, time: 87)
- channel space: (channel: 100, chromo: 2, time: 87)

---
vfc_hd

- parcel space: (time: 174, parcel: 145, chromo: 2)
- channel space: (channel: 214, wavelength: 2, time: 6712) -->


<!-- # Outdated approach -->

In [ ]:
# baseline_duration = 2.5  # in seconds
# n_shifts = 9
# duration = 10  # in seconds
# post_padding = 5  # in seconds
# n_timepoints = 87  # fixed length after shifting

# if DATASET_NAME == "BallSqueezingHD_modified":
#     delta_range = (-2.5, 2.5)
# elif DATASET_NAME == "FreshMotor":
#     delta_range = (-2.0, 0.0)
# start_shift = np.linspace(*delta_range, n_shifts)






# label_dict = {'right':1, 'left':2}  
# subject_to_rec = {}            
# INDEX = 0     
# freq_dir = processed_path / f'frq{0.5}'
# freq_dir.mkdir(exist_ok=True)       
# for file in proc_pkl_files:
#     with open(file, 'rb') as handle:
#         data_pickle = pickle.load(handle)
    
#     delta_brain = data_pickle['delta_conc']
#     sensitive_parcels = data_pickle['sensitive_parcels']
#     rec_stim = data_pickle['rec_stim']

#     # Align subject-specific parcels to a common parcel template (zero-pad missing parcels)
#     # delta_brain = delta_brain.sel(parcel=sensitive_parcels).reindex(parcel=PARCEL_TEMPLATE, fill_value=0)
#     delta_brain = delta_brain.sel(parcel=template_sens_parcel_list)

#     i = 0
    
    
    
    
#     # ------
#     SUB = PureWindowsPath(file).parts[-2]
#     if SUB not in subject_to_rec:
#         subject_to_rec[SUB] = []
#     try:
#         sub_dir = Path(freq_dir) / SUB
#         sub_dir.mkdir(parents=True, exist_ok=True)
#     except FileExistsError:
#         pass
#     # ------

#     for index, row in rec_stim.iterrows():
#         label = row["trial_type"].lower()
#         for s in start_shift:
#             start_time = row["onset"] + s
#             end_time = start_time + duration + post_padding # in seconds
#             baseline = delta_brain.sel(
#                 time=slice(row["onset"] - baseline_duration, row["onset"])
#             ).mean("time")
            
#             # Then, trimming is easy with `.sel()`:
#             x = delta_brain.sel(time=slice(start_time, end_time)) - baseline
#             x = x.isel(time=slice(0, n_timepoints))
#             x = x.transpose("parcel", "chromo", "time")
#             del x.time.attrs['units']

#             data = {
#                 'xt': x,
#                 'file': file,
#                 'class': label_dict[label],
#             }

#             # -----
#             # save events
#             filename = r'{}/{}/event_{}_delta{:+.2f}_{}.pkl'.format(freq_dir, SUB, 'aug' if s != 0.0 else 'orig', s, INDEX)
#             subject_to_rec[SUB].append(filename)
#             INDEX += 1
#             # -----
            
            
#             # if not os.path.exists(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path)))):
#             #     os.makedirs(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path))))
#             # if s == 0:
#             #     x.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+"_test.nc"))
#             #     i += 1
#             # else:
#             #     x.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+".nc"))
#             #     i += 1
#             with open(filename, 'wb') as handle:
#                 pickle.dump(data, handle, protocol=pickle.HIGHEST_PROTOCOL)
                
# with open('{}/meta_event_{}.pkl'.format(freq_dir, 0.5), 'wb') as handle:
#     pickle.dump(subject_to_rec, handle, protocol=pickle.HIGHEST_PROTOCOL)
# print('--- Done!---')            
#     # print("finished processing file: ", os.path.basename(file).replace(".pkl",".npy"))

In [ ]:
# events = str(processed_path / "frq{}" / "meta_event_{}.pkl")
    
# meta_events = []

# with open(events.format(0.5, 0.5), 'rb') as handle:
#     meta = pickle.load(handle)
# meta_events.append(meta)


# if DATASET_NAME == "BallSqueezingHD_modified":
#     session_to_files = {'run-1':[],
#                         'run-2':[],
#                         'run-3':[]}
    
# elif DATASET_NAME == "FreshMotor":    
#     session_to_files = {'run-left2s':[],
#                         'run-right2s':[],
#                         'run-left3s':[],
#                         'run-right3s':[]}

# files_to_session = {}
# for meta_event in meta_events:
#     for sub in meta_event:
#         for file in meta_event[sub]:
#             meta = None
#             with open(file, 'rb') as handle:
#                 meta = pickle.load(handle) 
            
#             # NEW: robust run parsing; safe with parameter-grid folder path layout
#             file_base = os.path.basename(meta['file'])
#             run_match = re.search(r"(run-[^_]+)", file_base)
#             if run_match is None:
#                 raise ValueError(f"Could not parse run token from file name: {file_base}")

#             run = run_match.group(1)
#             files_to_session[file] = run
#             if run not in session_to_files:
#                 session_to_files[run] = []
#             session_to_files[run].append(file)

# for run in session_to_files:
#     print(run, len(session_to_files[run]))

# # this will save the mapping of files to sessions used for LOSO
# with open(processed_path / 'files_to_sessions.pkl', 'wb') as handle:
#     pickle.dump(files_to_session, handle, protocol=pickle.HIGHEST_PROTOCOL)     
# print("Saved files_to_sessions.pkl")

In [ ]:
# # NEW: simple visualization for one recording across recon-parameter views
# # This cell expects you already ran the preprocessing cells and have:
# # rec, c_meas, Adot, get_recon_settings

# import matplotlib.pyplot as plt
# import numpy as np

# required_names = ["rec", "c_meas", "Adot", "get_recon_settings", "dot"]
# missing = [name for name in required_names if name not in globals()]
# if missing:
#     raise RuntimeError(f"Missing required variables for visualization: {missing}")

# # Reconstruct the same recording using all configured augmentation views.
# # This avoids surface-mesh plotting because brain_only=True returns only brain vertices.
# view_imgs = {}
# view_meta = {}
# for view_id, alpha_meas, alpha_spatial, alpha_meas_0, alpha_spatial_0, alpha_meas_multiplier, alpha_spatial_multiplier in get_recon_settings(c_meas):
#     recon_vis = dot.ImageRecon(
#         Adot,
#         recon_mode="mua2conc",
#         brain_only=True,
#         alpha_meas=float(alpha_meas),
#         alpha_spatial=float(alpha_spatial),
#         apply_c_meas=True,
#         spatial_basis_functions=None,
#     )
#     img = recon_vis.reconstruct(rec["od_pcr1"], c_meas)
#     img.time.attrs["units"] = units.s
#     img = img.cd.freq_filter(fmin=0.01, fmax=0.5, butter_order=4)
#     img = img.where(img.is_brain == True)
#     img = img.pint.to("uM").pint.dequantify()

#     view_imgs[view_id] = img
#     view_meta[view_id] = {
#         "alpha_meas": float(alpha_meas),
#         "alpha_spatial": float(alpha_spatial),
#         "alpha_meas_multiplier": float(alpha_meas_multiplier),
#         "alpha_spatial_multiplier": float(alpha_spatial_multiplier),
#     }

# if not view_imgs:
#     raise RuntimeError("No reconstruction views were generated.")

# # Choose one shared representative time from the middle view.
# ref_view = "am_1__as_1" if "am_1__as_1" in view_imgs else next(iter(view_imgs.keys()))
# ref_trace = view_imgs[ref_view].sel(chromo="HbO").mean("vertex", skipna=True)
# t_peak = ref_trace.time.values[np.nanargmax(np.abs(ref_trace.values))]

# fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(view_imgs)))

# # Panel 1: mean brain HbO time course per reconstruction setting.
# for color, (view_id, img) in zip(colors, view_imgs.items()):
#     meta = view_meta[view_id]
#     trace = img.sel(chromo="HbO").mean("vertex", skipna=True)
#     axes[0].plot(
#         trace.time.values,
#         trace.values,
#         color=color,
#         lw=2,
#         label=f"{view_id}: am={meta['alpha_meas']:.2e}, as={meta['alpha_spatial']:.1e}",
#     )
# axes[0].axvline(t_peak, color="k", lw=1, ls="--", alpha=0.6)
# axes[0].set_title("Mean brain HbO over time")
# axes[0].set_xlabel("time / s")
# axes[0].set_ylabel("HbO / uM")
# axes[0].grid(True, alpha=0.3)
# axes[0].legend(fontsize=8)

# # Panel 2: brain-vertex HbO distribution at the same peak time.
# labels = []
# distributions = []
# for view_id, img in view_imgs.items():
#     snap = img.sel(chromo="HbO").sel(time=t_peak, method="nearest")
#     vals = snap.values
#     vals = vals[np.isfinite(vals)]
#     labels.append(view_id)
#     distributions.append(vals)

# axes[1].boxplot(distributions, labels=labels, showfliers=False)
# axes[1].axhline(0, color="k", lw=1, alpha=0.5)
# axes[1].set_title(f"Brain-vertex HbO distribution at t={float(t_peak):.2f}s")
# axes[1].set_xlabel("reconstruction view")
# axes[1].set_ylabel("HbO / uM")
# axes[1].grid(True, axis="y", alpha=0.3)

# fig.suptitle("Effect of alpha_meas / alpha_spatial augmentation on one recording")
# fig.tight_layout()
# plt.show()


In [ ]:
# FINAL SUMMARY: dataset and segmentation settings actually used in this notebook
# Run this after preprocessing and segmentation cells.

from pathlib import Path
import os
import re
import numpy as np
import pandas as pd


def _safe_get(name, default=None):
    return globals().get(name, default)


def _subject_from_path(path):
    for part in Path(path).parts:
        if part.startswith("sub-"):
            return part
    return None


def _labels_from_segment_files(paths):
    labels = set()
    known_labels = ["left", "right", "task", "rest", "wordcongruent", "wordincongruent"]
    for path in paths:
        name = Path(path).name.lower()
        for label in known_labels:
            if f"_{label}_" in name:
                labels.add(label)
    return sorted(labels)


def _get_sampling_frequency():
    # Prefer the first raw file so this reports the dataset acquisition frequency.
    raw_files = _safe_get("files", [])
    if raw_files:
        try:
            rec0 = cedalion.io.read_snirf(raw_files[0])[0]
            t = np.asarray(rec0["amp"].time.values, dtype=float)
            if len(t) > 1:
                dt = float(np.median(np.diff(t)))
                return 1.0 / dt, dt
        except Exception as exc:
            print(f"Could not infer sampling frequency from raw file: {exc}")
    return None, None


def _get_first_segment_info(segment_files):
    if not segment_files:
        return None, None, None, None, None, None
    try:
        x = xr.open_dataarray(segment_files[0])
    except Exception:
        x = xr.open_dataset(segment_files[0]).to_array().squeeze()

    spatial_dim = "parcel" if "parcel" in x.dims else "channel" if "channel" in x.dims else None
    spatial_size = int(x.sizes[spatial_dim]) if spatial_dim else None
    chromo_size = int(x.sizes["chromo"]) if "chromo" in x.dims else None
    time_size = int(x.sizes["time"]) if "time" in x.dims else None

    segment_dt_seconds = None
    segment_fs_hz = None
    segment_duration_seconds = None
    if "time" in x.coords and time_size and time_size > 1:
        t = np.asarray(x.time.values, dtype=float)
        segment_dt_seconds = float(np.median(np.diff(t)))
        segment_fs_hz = 1.0 / segment_dt_seconds
        # Duration represented by samples. The coordinate span is one dt shorter.
        segment_duration_seconds = float(time_size * segment_dt_seconds)

    return (
        spatial_dim,
        spatial_size,
        chromo_size,
        time_size,
        segment_fs_hz,
        segment_dt_seconds,
        segment_duration_seconds,
    )


def _processed_raw_files(raw_files):
    # Match the notebook's processing exclusions before summarizing event timing.
    skipped = set(str(p) for p in _safe_get("skipped_subjects", []))
    dataset_name = _safe_get("DATASET_NAME")
    processed = []
    for file in raw_files:
        file = str(file)
        if file in skipped:
            continue
        if dataset_name == "vfc_hd" and "sub-13" in Path(file).parts:
            continue
        processed.append(file)
    return processed


def _get_min_event_onset_diff(raw_files):
    # Compute from the same event tables and standardization used by preprocessing.
    onset_diffs = []
    dataset_name = _safe_get("DATASET_NAME")
    for file in _processed_raw_files(raw_files):
        try:
            rec_i = cedalion.io.read_snirf(file)[0]
            stim_i = cedalion.io.read_events_from_tsv(file.replace("nirs.snirf", "events.tsv"))
            stim_i, rec_i = standardize_trial_types(dataset_name, file, stim_i, rec_i)
            onsets = np.sort(np.asarray(rec_i.stim.onset.values, dtype=float))
            if len(onsets) > 1:
                onset_diffs.extend(np.diff(onsets).tolist())
        except Exception as exc:
            print(f"Could not infer event onset spacing for {file}: {exc}")

    if not onset_diffs:
        return None, 0
    return float(np.min(onset_diffs)), len(onset_diffs)


raw_files = list(_safe_get("files", []))
processed_raw_files = _processed_raw_files(raw_files)
pre_path = Path(_safe_get("pre_processed_path")) if _safe_get("pre_processed_path") is not None else None
proc_path = Path(_safe_get("processed_path")) if _safe_get("processed_path") is not None else None

if proc_path is not None:
    segment_files = sorted(proc_path.glob("am_*__as_*/sub-*/*.nc"))
    if not segment_files:
        segment_files = sorted(proc_path.rglob("*.nc"))
else:
    segment_files = []

if pre_path is not None:
    preprocessed_files = sorted(pre_path.glob("am_*__as_*/sub-*/*.pkl"))
    if not preprocessed_files:
        preprocessed_files = sorted(pre_path.rglob("*.pkl"))
else:
    preprocessed_files = []

raw_subjects = sorted({s for s in (_subject_from_path(p) for p in raw_files) if s})
processed_subjects = sorted({Path(p).parent.name for p in segment_files if Path(p).parent.name.startswith("sub-")})
param_configs = sorted({part for p in segment_files for part in Path(p).parts if part.startswith("am_")})
labels = _labels_from_segment_files(segment_files)

fs_hz, dt_seconds = _get_sampling_frequency()
(
    spatial_dim,
    spatial_size_from_segment,
    chromo_size,
    time_size_from_segment,
    segment_fs_hz,
    segment_dt_seconds,
    segment_duration_seconds,
) = _get_first_segment_info(segment_files)

channel_size = len(_safe_get("subset_channels", [])) if _safe_get("subset_channels", None) is not None else None
parcel_template = _safe_get("template_sens_parcel_list", None)
parcel_size = len(parcel_template) if parcel_template is not None else None

n_times = _safe_get("n_timepoints", time_size_from_segment)
final_duration_seconds = segment_duration_seconds
if final_duration_seconds is None and n_times is not None:
    effective_fs = _safe_get("target_fs") if _safe_get("resample") else fs_hz
    final_duration_seconds = (float(n_times) / effective_fs) if effective_fs else None
min_event_onset_diff, n_event_onset_diffs = _get_min_event_onset_diff(raw_files)

summary = {
    "dataset_name": _safe_get("DATASET_NAME"),
    "augmentation_strategy": _safe_get("augmentation_strategy"),
    "subset_type": _safe_get("subset_type"),
    "raw_path": str(_safe_get("raw_path")),
    "pre_processed_path": str(pre_path),
    "processed_path": str(proc_path),
    "n_subjects_raw": len(raw_subjects),
    "n_subjects_processed": len(processed_subjects),
    "n_total_recordings_raw": len(raw_files),
    "n_total_recordings_processed_input": len(processed_raw_files),
    "n_preprocessed_recording_views": len(preprocessed_files),
    "n_segment_files_total": len(segment_files),
    "n_param_configs": len(param_configs),
    "param_configs": ", ".join(param_configs) if param_configs else "n/a",
    "channel_size": channel_size,
    "spatial_dim_in_segment": spatial_dim,
    "spatial_size_in_segment": spatial_size_from_segment,
    "parcel_template_size": parcel_size,
    "chromo_size": chromo_size,
    "raw_sampling_frequency_hz": f"{fs_hz:.3f}" if fs_hz else "n/a",
    "raw_sampling_dt_seconds": f"{dt_seconds:.6f}" if dt_seconds else "n/a",
    "resample": _safe_get("resample"),
    "target_fs_hz": _safe_get("target_fs"),
    "segment_sampling_frequency_hz": f"{segment_fs_hz:.3f}" if segment_fs_hz else "n/a",
    "segment_sampling_dt_seconds": f"{segment_dt_seconds:.6f}" if segment_dt_seconds else "n/a",
    "extract_rest_segments": _safe_get("extract_rest_segments"),
    "min_event_onset_diff_seconds": f"{min_event_onset_diff:.3f}" if min_event_onset_diff is not None else "n/a",
    "n_event_onset_diffs_used": n_event_onset_diffs,
    "n_times_per_segment_setting": n_times,
    "n_times_per_segment_from_file": time_size_from_segment,
    "final_segment_duration_seconds": f"{final_duration_seconds:.2f}" if final_duration_seconds else "n/a",
    "duration_setting_seconds": _safe_get("duration"),
    "post_padding_seconds": _safe_get("post_padding"),
    "baseline_duration_seconds": _safe_get("baseline_duration"),
    "delta_range": str(_safe_get("delta_range")),
    "number_of_shifts": _safe_get("n_shifts"),
    "start_shifts": np.array2string(_safe_get("start_shift"), precision=3) if _safe_get("start_shift", None) is not None else "n/a",
    "label_names": ", ".join(labels) if labels else "n/a",
}

summary_df = pd.DataFrame([summary]).T.reset_index()
summary_df.columns = ["field", "value"]

print("Final dataset / segmentation summary")
print("=" * 44)
display(summary_df)
# save data frame
summary_df.to_csv(proc_path / "dataset_segmentation_summary.csv", index=False)


Final dataset / segmentation summary


,field,value
0,dataset_name,vfc_hd
1,augmentation_strategy,imageRecon_params
2,subset_type,full
3,raw_path,datasets/raw/vfc_hd
4,pre_processed_path,datasets/pre_processed/imageRecon_params/vfc_h...
5,processed_path,datasets/processed/imageRecon_params/vfc_hd/full
6,n_subjects_raw,17
7,n_subjects_processed,16
8,n_total_recordings_raw,17
9,n_total_recordings_processed_input,16
